# 1. Tokenization

In [ ]:
import urllib.request

url =("https://raw.githubusercontent.com/yashaswi1306/Hindi_Language_Model/main/dataset_premchand.txt")
file_path="dataset_hindi.txt"

urllib.request.urlretrieve(url,file_path)

('dataset_hindi.txt', <http.client.HTTPMessage at 0x796255e14aa0>)

In [ ]:
with open("dataset_hindi.txt","r",encoding="utf-8") as f:
  data=f.read()
len(data)

48943

In [ ]:
import re

basic=re.split(r'([,.;:()?"_!\'])|\s',data)
basic_split=[]

for item in basic:
  if item is not None and item.strip():
    basic_split.append(item.strip())

# basic_split.extend(['<|unk|>','<|endoftext|>'])
len(basic_split), print(basic_split[:15])

['मेरे', 'भाई', 'साहब', 'मुझसे', 'पाँच', 'साल', 'बड़े', 'थे', ';', 'लेकिन', 'केवल', 'तीन', 'दरजे', 'आगे।', 'उन्होंने']


(10938, None)

In [ ]:
!wget https://raw.githubusercontent.com/yashaswi1306/Hindi_Language_Model/main/Byte_Pair_Encoder.ipynb

!jupyter nbconvert --to script Byte_Pair_Encoder.ipynb --stdout > Byte_Pair_Encoder.py

--2026-06-26 04:34:54--  https://raw.githubusercontent.com/yashaswi1306/Hindi_Language_Model/main/Byte_Pair_Encoder.ipynb
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6952 (6.8K) [text/plain]
Saving to: ‘Byte_Pair_Encoder.ipynb’

Byte_Pair_Encoder.i 100%[===================>]   6.79K  --.-KB/s    in 0s      

2026-06-26 04:34:54 (54.2 MB/s) - ‘Byte_Pair_Encoder.ipynb’ saved [6952/6952]

[NbConvertApp] Converting notebook Byte_Pair_Encoder.ipynb to script


In [ ]:
import os
print(os.listdir())

['.config', 'dataset_hindi.txt', 'Byte_Pair_Encoder.ipynb', 'Byte_Pair_Encoder.py', 'sample_data']


In [ ]:
from Byte_Pair_Encoder import BPE_Encoder, get_stats,merge

tokenizer=BPE_Encoder()
tokenizer.train(data,vocab_size=600)
tokenizer.reg_special_tokens({"<|endoftext|>": 799})

In [ ]:
print(f"Vocab Size : {len(tokenizer.vocab)}")
print(f"Merges : {len(tokenizer.merges)}")

sample="प्रेमचंद की कथाओ पर आधारित।"
ids=tokenizer.encode(sample)
decode=tokenizer.decode(ids)

print(f"Original text: {sample}")
print(f"Encoded: {ids}")
print(f"Decoded: {decode}")

if (sample==decode):
  print("Successfully decoded")
else:
  print("Unsuccessfull")


sample2="प्रेमचंद की कथाओ पर आधारित।<|endoftext|>बड़े भाई साहब व अन्य कहानियाँ।"
ids=tokenizer.encode(sample2,allowed_special={"<|endoftext|>"}) # allowed special MUST be a set NOT a a string
decode=tokenizer.decode(ids)

print(f"Original text: {sample2}")
print(f"Encoded: {ids}")
print(f"Decoded: {decode}")

if (sample2==decode):
  print("Successfully decoded")
else:
  print("Unsuccessfull")


Vocab Size : 601
Merges : 344
Original text: प्रेमचंद की कथाओ पर आधारित।
Encoded: [311, 336, 260, 288, 322, 266, 301, 267, 332, 256, 165, 261, 147, 377, 134, 366, 293, 363, 276]
Decoded: प्रेमचंद की कथाओ पर आधारित।
Successfully decoded
Original text: प्रेमचंद की कथाओ पर आधारित।<|endoftext|>बड़े भाई साहब व अन्य कहानियाँ।
Encoded: [311, 336, 260, 288, 322, 266, 301, 267, 332, 256, 165, 261, 147, 377, 134, 366, 293, 363, 276, 799, 297, 258, 156, 264, 529, 589, 389, 268, 319, 267, 274, 328, 513, 276]
Decoded: प्रेमचंद की कथाओ पर आधारित।<|endoftext|>बड़े भाई साहब व अन्य कहानियाँ।
Successfully decoded


In [ ]:
ids[:10] # sliding window pairs needed for the model with input and target
# toh if a pair is (a,b), (c,d) input=[a,c] target=[b,d]

[311, 336, 260, 288, 322, 266, 301, 267, 332, 256]

# Data Loader

In [ ]:
import torch
from torch.utils.data import Dataset,DataLoader

class Hindi_Dataset(Dataset):
  def __init__(self,tokenizer,text,context_window,stride):

    tokens=tokenizer.encode(text)
    self.input=[]
    self.target=[]

    for i in range(0, len(tokens)-context_window,stride):
      self.input.append(torch.tensor(tokens[i:i+context_window]))
      self.target.append(torch.tensor(tokens[i+1:i+context_window+1]))

  def __len__(self):
    return len(self.input)

  def __getitem__(self, index):
    return self.input[index], self.target[index]

In [ ]:
context_window=64
stride=32
batch_size=8

dataset=Hindi_Dataset(tokenizer,data,context_window,stride)
dataloader=DataLoader(dataset,batch_size,shuffle=True)

x,y=next(iter(dataloader)) # iterate ove dataloader

x.shape,y.shape,x[0],y[0] # first batch

(torch.Size([8, 64]),
 torch.Size([8, 64]),
 tensor([174, 372, 451, 303, 172, 433, 478, 293, 264, 166, 342, 267, 428, 307,
         328, 332, 286, 323, 168, 307, 375, 155, 275, 259, 598, 257, 537, 182,
         336, 301, 538, 270, 133, 459, 164, 273, 424, 319, 535,  45, 273, 435,
         275, 534, 165, 371, 537, 166, 258, 131, 256, 183, 539, 269, 356, 349,
         389, 297, 469, 566, 322, 434, 464, 185]),
 tensor([372, 451, 303, 172, 433, 478, 293, 264, 166, 342, 267, 428, 307, 328,
         332, 286, 323, 168, 307, 375, 155, 275, 259, 598, 257, 537, 182, 336,
         301, 538, 270, 133, 459, 164, 273, 424, 319, 535,  45, 273, 435, 275,
         534, 165, 371, 537, 166, 258, 131, 256, 183, 539, 269, 356, 349, 389,
         297, 469, 566, 322, 434, 464, 185, 497]))

In [ ]:
len(dataloader)

98

# Attention

In [ ]:
import torch
import torch.nn as nn

In [ ]:
class Simple_Self_Attention(nn.Module): # K,Q and V are all X
  def forward(self,x):
    attn_scores=x@x.transpose(-2,-1) # so batch, token,embedding -> batch, embedding, token
    attn_weights=torch.softmax(attn_scores,dim=-1) # softmax across each row

    vec_context=attn_weights@x
    return vec_context,attn_weights

In [ ]:
x=torch.tensor([
    [
        [1,0],
        [0,1],
        [1,1]
    ]
])

x=x.type(torch.float)
x.shape

torch.Size([1, 3, 2])

In [ ]:
model_simple=Simple_Self_Attention()
context,weights=model_simple(x)

weights

tensor([[[0.4223, 0.1554, 0.4223],
         [0.1554, 0.4223, 0.4223],
         [0.2119, 0.2119, 0.5761]]])

In [ ]:
class Multi_Head_Attention(nn.Module):
  def __init__(self,dim_in,dim_out,context_window,dropout,num_heads,bias):
    super().__init__()
    assert dim_out%num_heads==0
    self.d_out=dim_out
    self.num_heads=num_heads

    self.head_dim=dim_out//num_heads

    # create dim_in*dim_out matrix for q,k,v
    self.query=nn.Linear(dim_in,dim_out,bias)
    self.key=nn.Linear(dim_in,dim_out,bias)
    self.value=nn.Linear(dim_in,dim_out,bias)

    self.dropout=nn.Dropout(dropout) # randomised dropout to prevent over-concentration
    self.out=nn.Linear(dim_out,dim_out) #allows the vectors for diff tokens to be interdependent

    self.register_buffer("mask",torch.triu(torch.ones(context_window,context_window),diagonal=1)) # mask for upper triangle matrix with ones

    def forward(self,x):
      batch,token_num,dim_in=x.shape
      Q=self.query(x) # Q=x@q+ b
      K=self.key(x)
      V=self.value(x)

      Q=Q.reshape(batch,token_num,self.num_heads,self.head_dim).transpose(1,2)
      K=K.reshape(batch,token_num,self.num_heads,self.head_dim).transpose(1,2)
      V=V.reshape(batch,token_num,self.num_heads,self.head_dim).transpose(1,2)

      # instead of:(batch,tokens,heads,head_dim)
      # Token1:
      # Head1
      # Head2
      # Head3
      # we want:(batch,tokens,heads,head_dim)
      # Head1:
      # Token1
      # Token2
      # Token3

      attn_scores=Q@K.transpose(-2,-1) # so batch, token,embedding -> batch, embedding, token
      attn_scores=attn_scores/(self.head_dim**0.5) # If vecs too large then softmax peaked cuz dot product too high.
      # Hence, we divide by the root of head dim
      attn_scores=attn_scores.masked_fill(self.mask[:token_num,:token_num].bool(),float("-inf"))

      attn_weights=torch.softmax(attn_scores,dim=-1) # softmax across each row
      attn_weights=torch.dropout(attn_weights)

      vec_context=attn_weights@V # -> (batch,heads,tokens,head_dim)
      vec_context=vec_context.transpose(1,2) # -> (batch,tokens,heads,head_dim)
      vec_context=vec_context.contiguous() # so no issues in .view

      vec_context=vec_context.reshape(batch,token_num,self.d_out) # -> (batch,tokens,heads*head_dim)

      return self.out(vec_context) # output = context_vecs @ W + b



In [26]:
model_mh=Multi_Head_Attention(4,4,5,0,2,1)
context=model_simple(x)

context

(tensor([[[0.8446, 0.5777],
          [0.5777, 0.8446],
          [0.7881, 0.7881]]]),
 tensor([[[0.4223, 0.1554, 0.4223],
          [0.1554, 0.4223, 0.4223],
          [0.2119, 0.2119, 0.5761]]]))